In [2]:
import os
import pandas as pd
from tqdm import tqdm
tqdm.pandas() 
folder_path = '/Users/andrey/Desktop/MisisProjects/tasks-public/Arenadata/telecom100k'

In [7]:
# Получаем список всех файлов в папке, которые начинаются с 'psx_'
files = [f for f in os.listdir(folder_path) if f.startswith('psx_')]

# Создаем пустой DataFrame для объединения всех данных
combined_df = pd.DataFrame()

# Проходим по каждому файлу и добавляем его в общий DataFrame
for file in tqdm(files):
    file_path = os.path.join(folder_path, file)
    # Читаем файл в DataFrame
    df = pd.read_csv(file_path)
    # Добавляем колонку с именем файла
    df['name_data'] = file
    # Объединяем с общим DataFrame
    combined_df = pd.concat([combined_df, df], ignore_index=True)

# Выводим результат
print(combined_df.info())

100%|██████████| 6048/6048 [14:19<00:00,  7.04it/s]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11124648 entries, 0 to 11124647
Data columns (total 10 columns):
 #   Column                                                                     Dtype  
---  ------                                                                     -----  
 0   IdSession|IdPSX|IdSubscriber|StartSession|EndSession|Duartion|UpTx|DownTx  object 
 1   name_data                                                                  object 
 2   IdSession                                                                  float64
 3   IdPSX                                                                      float64
 4   IdSubscriber                                                               float64
 5   StartSession                                                               object 
 6   EndSession                                                                 object 
 7   Duartion                                                                   float64
 8   

In [8]:
split_data = combined_df[combined_df['IdSession|IdPSX|IdSubscriber|StartSession|EndSession|Duartion|UpTx|DownTx'].notna()]['IdSession|IdPSX|IdSubscriber|StartSession|EndSession|Duartion|UpTx|DownTx'].str.split('|', expand=True).rename(columns={0: 'IdSession', 1: 'IdPSX', 2: 'IdSubscriber', 3: 'StartSession', 4: 'EndSession', 5: 'Duartion', 6: 'UpTx', 7: 'DownTx'})
split_data['name_data'] = combined_df[combined_df['IdSession|IdPSX|IdSubscriber|StartSession|EndSession|Duartion|UpTx|DownTx'].notna()]['name_data']
new_combined_df = pd.concat([combined_df[combined_df['IdSession|IdPSX|IdSubscriber|StartSession|EndSession|Duartion|UpTx|DownTx'].isna()].drop(columns=['IdSession|IdPSX|IdSubscriber|StartSession|EndSession|Duartion|UpTx|DownTx']), split_data], ignore_index=True)

In [9]:
split_data.isna().sum()

IdSession       0
IdPSX           0
IdSubscriber    0
StartSession    0
EndSession      0
Duartion        0
UpTx            0
DownTx          0
name_data       0
dtype: int64

In [10]:
combined_df[combined_df['IdSession|IdPSX|IdSubscriber|StartSession|EndSession|Duartion|UpTx|DownTx'].isna()].drop(columns=['IdSession|IdPSX|IdSubscriber|StartSession|EndSession|Duartion|UpTx|DownTx']).isna().sum()

name_data             0
IdSession             0
IdPSX                 0
IdSubscriber          0
StartSession          0
EndSession      5404638
Duartion              0
UpTx                  0
DownTx                0
dtype: int64

In [11]:
new_combined_df.isna().sum()

name_data             0
IdSession             0
IdPSX                 0
IdSubscriber          0
StartSession          0
EndSession      5404638
Duartion              0
UpTx                  0
DownTx                0
dtype: int64

In [12]:
len(new_combined_df) == len(combined_df)

True

In [13]:
new_combined_df.head()

,name_data,IdSession,IdPSX,IdSubscriber,StartSession,EndSession,Duartion,UpTx,DownTx
0,psx_65.0_2024-01-04 19:20:00.csv,22324.0,5.0,6529.0,04-01-2024 12:35:35,NaN,24265.0,1480800364.0,1481794943.0
1,psx_65.0_2024-01-04 19:20:00.csv,20535.0,5.0,29242.0,04-01-2024 00:09:40,04-01-2024 19:17:12,68852.0,1760236098.0,2530665408.0
2,psx_65.0_2024-01-04 19:20:00.csv,23238.0,5.0,29242.0,04-01-2024 19:18:52,NaN,68.0,1377482.0,1943318.0
3,psx_65.0_2024-01-04 19:20:00.csv,22857.0,5.0,79675.0,04-01-2024 16:27:17,NaN,10363.0,117643980.0,129768393.0
4,psx_65.0_2024-01-04 19:20:00.csv,22814.0,5.0,45133.0,04-01-2024 16:02:31,NaN,11849.0,555253055.0,781371661.0


In [14]:
new_combined_df['EndSession'] = new_combined_df.apply(
        lambda row: row['name_data'].split('_')[2][:-4] if pd.isna(row['EndSession']) else row['EndSession'], axis=1
    )

In [15]:
new_combined_df['psx_name'] = new_combined_df['name_data'].apply(lambda x: '_'.join(x.split('_')[:2]))

In [16]:
new_combined_df['PSX'] = new_combined_df['psx_name'].str.split('_', expand=True)[1].astype(float)

In [17]:
new_combined_df

,name_data,IdSession,IdPSX,IdSubscriber,StartSession,EndSession,Duartion,UpTx,DownTx,psx_name,PSX
0,psx_65.0_2024-01-04 19:20:00.csv,22324.0,5.0,6529.0,04-01-2024 12:35:35,2024-01-04 19:20:00,24265.0,1480800364.0,1481794943.0,psx_65.0,65.0
1,psx_65.0_2024-01-04 19:20:00.csv,20535.0,5.0,29242.0,04-01-2024 00:09:40,04-01-2024 19:17:12,68852.0,1760236098.0,2530665408.0,psx_65.0,65.0
2,psx_65.0_2024-01-04 19:20:00.csv,23238.0,5.0,29242.0,04-01-2024 19:18:52,2024-01-04 19:20:00,68.0,1377482.0,1943318.0,psx_65.0,65.0
3,psx_65.0_2024-01-04 19:20:00.csv,22857.0,5.0,79675.0,04-01-2024 16:27:17,2024-01-04 19:20:00,10363.0,117643980.0,129768393.0,psx_65.0,65.0
4,psx_65.0_2024-01-04 19:20:00.csv,22814.0,5.0,45133.0,04-01-2024 16:02:31,2024-01-04 19:20:00,11849.0,555253055.0,781371661.0,psx_65.0,65.0
...,...,...,...,...,...,...,...,...,...,...,...
11124643,psx_66.3_2024-01-05 14:50:00.txt,22775,2,15606,05/01/2024 12:21:26,,8914,2574876440,3719874912,psx_66.3,66.3
11124644,psx_66.3_2024-01-05 14:50:00.txt,21244,2,39993,05/01/2024 03:35:14,,40486,29807975016,29883569080,psx_66.3,66.3
11124645,psx_66.3_2024-01-05 14:50:00.txt,21992,2,54026,05/01/2024 07:55:42,,24858,1170151648,1183538544,psx_66.3,66.3
11124646,psx_66.3_2024-01-05 14:50:00.txt,19853,2,19194,04/01/2024 19:44:43,,68717,14778347296,17004107904,psx_66.3,66.3


In [18]:
new_combined_df.isna().sum()

name_data       0
IdSession       0
IdPSX           0
IdSubscriber    0
StartSession    0
EndSession      0
Duartion        0
UpTx            0
DownTx          0
psx_name        0
PSX             0
dtype: int64

In [19]:
new_combined_df.isna().sum()

name_data       0
IdSession       0
IdPSX           0
IdSubscriber    0
StartSession    0
EndSession      0
Duartion        0
UpTx            0
DownTx          0
psx_name        0
PSX             0
dtype: int64

In [20]:
new_combined_df.to_csv('new_combined_df.csv', index=False)

In [21]:
new_combined_df = pd.read_csv('new_combined_df.csv')

In [22]:
new_combined_df

,name_data,IdSession,IdPSX,IdSubscriber,StartSession,EndSession,Duartion,UpTx,DownTx,psx_name,PSX
0,psx_65.0_2024-01-04 19:20:00.csv,22324.0,5.0,6529.0,04-01-2024 12:35:35,2024-01-04 19:20:00,24265.0,1.480800e+09,1.481795e+09,psx_65.0,65.0
1,psx_65.0_2024-01-04 19:20:00.csv,20535.0,5.0,29242.0,04-01-2024 00:09:40,04-01-2024 19:17:12,68852.0,1.760236e+09,2.530665e+09,psx_65.0,65.0
2,psx_65.0_2024-01-04 19:20:00.csv,23238.0,5.0,29242.0,04-01-2024 19:18:52,2024-01-04 19:20:00,68.0,1.377482e+06,1.943318e+06,psx_65.0,65.0
3,psx_65.0_2024-01-04 19:20:00.csv,22857.0,5.0,79675.0,04-01-2024 16:27:17,2024-01-04 19:20:00,10363.0,1.176440e+08,1.297684e+08,psx_65.0,65.0
4,psx_65.0_2024-01-04 19:20:00.csv,22814.0,5.0,45133.0,04-01-2024 16:02:31,2024-01-04 19:20:00,11849.0,5.552531e+08,7.813717e+08,psx_65.0,65.0
...,...,...,...,...,...,...,...,...,...,...,...
11124643,psx_66.3_2024-01-05 14:50:00.txt,22775.0,2.0,15606.0,05/01/2024 12:21:26,NaN,8914.0,2.574876e+09,3.719875e+09,psx_66.3,66.3
11124644,psx_66.3_2024-01-05 14:50:00.txt,21244.0,2.0,39993.0,05/01/2024 03:35:14,NaN,40486.0,2.980798e+10,2.988357e+10,psx_66.3,66.3
11124645,psx_66.3_2024-01-05 14:50:00.txt,21992.0,2.0,54026.0,05/01/2024 07:55:42,NaN,24858.0,1.170152e+09,1.183539e+09,psx_66.3,66.3
11124646,psx_66.3_2024-01-05 14:50:00.txt,19853.0,2.0,19194.0,04/01/2024 19:44:43,NaN,68717.0,1.477835e+10,1.700411e+10,psx_66.3,66.3


In [23]:
psxattrs = pd.read_csv(folder_path + '/' + 'psxattrs.csv')

In [24]:
merged_df = pd.merge(new_combined_df, psxattrs)

In [25]:
merged_df['DownTx'] = merged_df.apply(lambda row: row['DownTx'] * 8 if row['TransmitUnits'] == 'bytes' else row['DownTx'], axis=1)
merged_df['UpTx'] = merged_df.apply(lambda row: row['UpTx'] * 8 if row['TransmitUnits'] == 'bytes' else row['UpTx'], axis=1)

In [26]:
psxattrs

,Id,PSX,TransmitUnits,Delimiter,DateFormat,TZ
0,0,66.1,bits,|,%d/%m/%Y %H:%M:%S,GMT-5
1,1,66.2,bits,|,%d/%m/%Y %H:%M:%S,GMT-5
2,2,66.3,bits,|,%d/%m/%Y %H:%M:%S,GMT-5
3,3,62.0,bytes,",",%d-%m-%Y %H:%M:%S,GMT-6
4,4,69.0,bytes,",",%d-%m-%Y %H:%M:%S,GMT-6
5,5,65.0,bytes,",",%d-%m-%Y %H:%M:%S,GMT-6


In [27]:
merged_df.isna().sum()

name_data              0
IdSession              0
IdPSX                  0
IdSubscriber           0
StartSession           0
EndSession       5545646
Duartion               0
UpTx                   0
DownTx                 0
psx_name               0
PSX                    0
Id                     0
TransmitUnits          0
Delimiter              0
DateFormat             0
TZ                     0
dtype: int64

In [33]:
import pandas as pd
from datetime import datetime
import pytz

def convert_sessions_to_gmt5(df):
    def convert_row(row):
        # Determine the timezone offset
        timezone_offset = row['TZ']
        timezone = pytz.timezone('Etc/' + str(timezone_offset))
        
        # Convert StartSession
        try:
            start_session = datetime.strptime(row['StartSession'], row['DateFormat'])
            start_session = timezone.localize(start_session).astimezone(pytz.timezone('Etc/GMT+5'))
            row['StartSession'] = start_session.strftime('%d-%m-%Y %H:%M:%S')
        except ValueError as e:
            print(f"Error parsing StartSession for row: {row} - {e}")
        
        # Convert EndSession
        try:
            if isinstance(row['EndSession'], str):
                # Check if the year is at the beginning
                if row['EndSession'].startswith('20'):
                    # Assume format is '%Y-%m-%d %H:%M:%S'
                    end_session = datetime.strptime(row['EndSession'], '%Y-%m-%d %H:%M:%S')
                else:
                    end_session = datetime.strptime(row['EndSession'], row['DateFormat'])
            else:
                # If EndSession is None, assume the default format
                end_session = datetime.strptime(row['StartSession'], '%d-%m-%Y %H:%M:%S')
            end_session = timezone.localize(end_session).astimezone(pytz.timezone('Etc/GMT+5'))
            row['EndSession'] = end_session.strftime('%d-%m-%Y %H:%M:%S')
        except ValueError as e:
            print(f"Error parsing EndSession for row: {row} - {e}")
        
        return row

    # Apply the conversion function to each row
    df = df.progress_apply(convert_row, axis=1)
    return df

# Example usage
# df = pd.DataFrame({
#     'StartSession': ['01/12/2023 14:30:00', '02-12-2023 15:45:00'],
#     'EndSession': [None, '2023-12-02 17:45:00'],
#     'DateFormat': ['%d/%m/%Y %H:%M:%S', '%d-%m-%Y %H:%M:%S'],
#     'TZ': [-5, -6]
# })

# df = convert_sessions_to_gmt5(df)
# print(df)

In [34]:
import gc
gc.collect()

402

In [41]:
merged_df['EndSession'].isna().sum()

np.int64(0)

In [42]:
merged_df['EndSession'] = merged_df['EndSession'].apply(lambda x:  x.split('.')[0] if isinstance(x, str) else x)
merged_df['EndSession']

0           04-01-2024 08:20:00
1           04-01-2024 08:17:12
2           04-01-2024 08:20:00
3           04-01-2024 08:20:00
4           04-01-2024 08:20:00
                   ...         
11124643    04-01-2024 16:21:26
11124644    04-01-2024 07:35:14
11124645    04-01-2024 11:55:42
11124646    03-01-2024 23:44:43
11124647    04-01-2024 02:11:12
Name: EndSession, Length: 11124648, dtype: object

In [37]:
merged_df = convert_sessions_to_gmt5(merged_df)
merged_df

100%|██████████| 11124648/11124648 [10:30<00:00, 17657.78it/s]


,name_data,IdSession,IdPSX,IdSubscriber,StartSession,EndSession,Duartion,UpTx,DownTx,psx_name,PSX,Id,TransmitUnits,Delimiter,DateFormat,TZ
0,psx_65.0_2024-01-04 19:20:00.csv,22324.0,5.0,6529.0,04-01-2024 01:35:35,04-01-2024 08:20:00,24265.0,1.184640e+10,1.185436e+10,psx_65.0,65.0,5,bytes,",",%d-%m-%Y %H:%M:%S,GMT-6
1,psx_65.0_2024-01-04 19:20:00.csv,20535.0,5.0,29242.0,03-01-2024 13:09:40,04-01-2024 08:17:12,68852.0,1.408189e+10,2.024532e+10,psx_65.0,65.0,5,bytes,",",%d-%m-%Y %H:%M:%S,GMT-6
2,psx_65.0_2024-01-04 19:20:00.csv,23238.0,5.0,29242.0,04-01-2024 08:18:52,04-01-2024 08:20:00,68.0,1.101986e+07,1.554654e+07,psx_65.0,65.0,5,bytes,",",%d-%m-%Y %H:%M:%S,GMT-6
3,psx_65.0_2024-01-04 19:20:00.csv,22857.0,5.0,79675.0,04-01-2024 05:27:17,04-01-2024 08:20:00,10363.0,9.411518e+08,1.038147e+09,psx_65.0,65.0,5,bytes,",",%d-%m-%Y %H:%M:%S,GMT-6
4,psx_65.0_2024-01-04 19:20:00.csv,22814.0,5.0,45133.0,04-01-2024 05:02:31,04-01-2024 08:20:00,11849.0,4.442024e+09,6.250973e+09,psx_65.0,65.0,5,bytes,",",%d-%m-%Y %H:%M:%S,GMT-6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11124643,psx_66.3_2024-01-05 14:50:00.txt,22775.0,2.0,15606.0,05-01-2024 02:21:26,04-01-2024 16:21:26,8914.0,2.574876e+09,3.719875e+09,psx_66.3,66.3,2,bits,|,%d/%m/%Y %H:%M:%S,GMT-5
11124644,psx_66.3_2024-01-05 14:50:00.txt,21244.0,2.0,39993.0,04-01-2024 17:35:14,04-01-2024 07:35:14,40486.0,2.980798e+10,2.988357e+10,psx_66.3,66.3,2,bits,|,%d/%m/%Y %H:%M:%S,GMT-5
11124645,psx_66.3_2024-01-05 14:50:00.txt,21992.0,2.0,54026.0,04-01-2024 21:55:42,04-01-2024 11:55:42,24858.0,1.170152e+09,1.183539e+09,psx_66.3,66.3,2,bits,|,%d/%m/%Y %H:%M:%S,GMT-5
11124646,psx_66.3_2024-01-05 14:50:00.txt,19853.0,2.0,19194.0,04-01-2024 09:44:43,03-01-2024 23:44:43,68717.0,1.477835e+10,1.700411e+10,psx_66.3,66.3,2,bits,|,%d/%m/%Y %H:%M:%S,GMT-5


In [43]:
merged_df.to_csv('merged_df.csv', index=False)

In [3]:
client = pd.read_parquet(folder_path +'/'+'client.parquet')
company = pd.read_parquet(folder_path +'/'+'company.parquet')
physical = pd.read_parquet(folder_path +'/'+'physical.parquet')
subscribers = pd.read_csv(folder_path + '/' + 'subscribers.csv')

In [4]:
subscribers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96208 entries, 0 to 96207
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   IdClient  96208 non-null  object
 1   IdOnPSX   96208 non-null  int64 
 2   Status    96208 non-null  object
dtypes: int64(1), object(2)
memory usage: 2.2+ MB


In [46]:
subscribers_merged = subscribers.merge(client, left_on="IdClient", right_on="Id")

In [47]:
subscribers_merged['IdPlan']

0        4
1        3
2        2
3        2
4        3
        ..
96203    3
96204    4
96205    4
96206    7
96207    3
Name: IdPlan, Length: 96208, dtype: int64

In [48]:
company_id = set(company['Id'].unique().tolist())

In [30]:
subscribers_merged['Type'] = subscribers_merged['Id'].progress_apply(lambda x: "C" if x in company_id else "P")
subscribers_merged['Type'].value_counts()

100%|██████████| 96208/96208 [00:00<00:00, 2689095.02it/s]


Type
P    84791
C    11417
Name: count, dtype: int64

In [40]:
merged_df.isna().sum()

name_data        0
IdSession        0
IdPSX            0
IdSubscriber     0
StartSession     0
EndSession       0
Duartion         0
UpTx             0
DownTx           0
psx_name         0
PSX              0
Id               0
TransmitUnits    0
Delimiter        0
DateFormat       0
TZ               0
dtype: int64

In [50]:
needed_df = merged_df[['IdSession', 'IdPSX', 'IdSubscriber', 'StartSession', 'EndSession', 'Duartion', 'UpTx', 'DownTx']]

In [51]:
subscribers_merged = pd.read_csv('subscribers_merged.csv')

In [52]:
subscribers_merged

,IdClient,IdOnPSX,Status,Id,Contract,Documents,Email,IdPlan,Type
0,b99ef4f4-9429-428e-8c81-7cdc2dbc320d,3792,ON,b99ef4f4-9429-428e-8c81-7cdc2dbc320d,GB77ABGD33,internal.store.com/clients/documents/GB77ABGD33,dadams@rivera.com,4,P
1,98d1864e-0be8-4532-a8f2-2182cf84094b,3793,OFF,98d1864e-0be8-4532-a8f2-2182cf84094b,GB30BRPQ96,internal.store.com/clients/documents/GB30BRPQ96,francogerald@miller.biz,3,P
2,4f752c49-49e8-4695-ac5e-f90e7907c425,3794,ON,4f752c49-49e8-4695-ac5e-f90e7907c425,GB66SHAM82,internal.store.com/clients/documents/GB66SHAM82,michaelortiz@golden.com,2,P
3,2884b847-8b92-4e6c-8d33-309699315105,3795,ON,2884b847-8b92-4e6c-8d33-309699315105,GB18CFEL59,internal.store.com/clients/documents/GB18CFEL59,cheyenne83@pena.biz,2,P
4,8826c5cf-6943-4e7a-8601-ca6adae44a1d,3796,ON,8826c5cf-6943-4e7a-8601-ca6adae44a1d,GB53YSST03,internal.store.com/clients/documents/GB53YSST03,jessica61@gonzalez-wong.com,3,P
...,...,...,...,...,...,...,...,...,...
96203,684f155e-6eee-451e-a3c6-94711f3c522d,99995,ON,684f155e-6eee-451e-a3c6-94711f3c522d,GB18LSRH18,internal.store.com/clients/documents/GB18LSRH18,crawfordjohn@williams.info,3,P
96204,d0f4199e-b964-42b2-874b-c7fbb260d1e7,99996,ON,d0f4199e-b964-42b2-874b-c7fbb260d1e7,GB17OOFG98,internal.store.com/clients/documents/GB17OOFG98,debbie34@duncan-nelson.info,4,P
96205,4a728c88-2441-4fce-aa9a-1b99a67a8564,99997,ON,4a728c88-2441-4fce-aa9a-1b99a67a8564,GB55FRUX19,internal.store.com/clients/documents/GB55FRUX19,nallen@warner-mccarthy.com,4,P
96206,0c3cc209-ba06-4fb2-bd6e-f19635dd2fc5,99998,ON,0c3cc209-ba06-4fb2-bd6e-f19635dd2fc5,GB09LWJL63,internal.store.com/clients/documents/GB09LWJL63,eric78@watson.info,7,P


In [53]:
needed_df

,IdSession,IdPSX,IdSubscriber,StartSession,EndSession,Duartion,UpTx,DownTx
0,22324.0,5.0,6529.0,04-01-2024 01:35:35,04-01-2024 08:20:00,24265.0,1.184640e+10,1.185436e+10
1,20535.0,5.0,29242.0,03-01-2024 13:09:40,04-01-2024 08:17:12,68852.0,1.408189e+10,2.024532e+10
2,23238.0,5.0,29242.0,04-01-2024 08:18:52,04-01-2024 08:20:00,68.0,1.101986e+07,1.554654e+07
3,22857.0,5.0,79675.0,04-01-2024 05:27:17,04-01-2024 08:20:00,10363.0,9.411518e+08,1.038147e+09
4,22814.0,5.0,45133.0,04-01-2024 05:02:31,04-01-2024 08:20:00,11849.0,4.442024e+09,6.250973e+09
...,...,...,...,...,...,...,...,...
11124643,22775.0,2.0,15606.0,05-01-2024 02:21:26,04-01-2024 16:21:26,8914.0,2.574876e+09,3.719875e+09
11124644,21244.0,2.0,39993.0,04-01-2024 17:35:14,04-01-2024 07:35:14,40486.0,2.980798e+10,2.988357e+10
11124645,21992.0,2.0,54026.0,04-01-2024 21:55:42,04-01-2024 11:55:42,24858.0,1.170152e+09,1.183539e+09
11124646,19853.0,2.0,19194.0,04-01-2024 09:44:43,03-01-2024 23:44:43,68717.0,1.477835e+10,1.700411e+10


In [55]:
subscribers_merged

,IdClient,IdOnPSX,Status,Id,Contract,Documents,Email,IdPlan,Type
0,b99ef4f4-9429-428e-8c81-7cdc2dbc320d,3792,ON,b99ef4f4-9429-428e-8c81-7cdc2dbc320d,GB77ABGD33,internal.store.com/clients/documents/GB77ABGD33,dadams@rivera.com,4,P
1,98d1864e-0be8-4532-a8f2-2182cf84094b,3793,OFF,98d1864e-0be8-4532-a8f2-2182cf84094b,GB30BRPQ96,internal.store.com/clients/documents/GB30BRPQ96,francogerald@miller.biz,3,P
2,4f752c49-49e8-4695-ac5e-f90e7907c425,3794,ON,4f752c49-49e8-4695-ac5e-f90e7907c425,GB66SHAM82,internal.store.com/clients/documents/GB66SHAM82,michaelortiz@golden.com,2,P
3,2884b847-8b92-4e6c-8d33-309699315105,3795,ON,2884b847-8b92-4e6c-8d33-309699315105,GB18CFEL59,internal.store.com/clients/documents/GB18CFEL59,cheyenne83@pena.biz,2,P
4,8826c5cf-6943-4e7a-8601-ca6adae44a1d,3796,ON,8826c5cf-6943-4e7a-8601-ca6adae44a1d,GB53YSST03,internal.store.com/clients/documents/GB53YSST03,jessica61@gonzalez-wong.com,3,P
...,...,...,...,...,...,...,...,...,...
96203,684f155e-6eee-451e-a3c6-94711f3c522d,99995,ON,684f155e-6eee-451e-a3c6-94711f3c522d,GB18LSRH18,internal.store.com/clients/documents/GB18LSRH18,crawfordjohn@williams.info,3,P
96204,d0f4199e-b964-42b2-874b-c7fbb260d1e7,99996,ON,d0f4199e-b964-42b2-874b-c7fbb260d1e7,GB17OOFG98,internal.store.com/clients/documents/GB17OOFG98,debbie34@duncan-nelson.info,4,P
96205,4a728c88-2441-4fce-aa9a-1b99a67a8564,99997,ON,4a728c88-2441-4fce-aa9a-1b99a67a8564,GB55FRUX19,internal.store.com/clients/documents/GB55FRUX19,nallen@warner-mccarthy.com,4,P
96206,0c3cc209-ba06-4fb2-bd6e-f19635dd2fc5,99998,ON,0c3cc209-ba06-4fb2-bd6e-f19635dd2fc5,GB09LWJL63,internal.store.com/clients/documents/GB09LWJL63,eric78@watson.info,7,P


In [58]:
final_df = needed_df.merge(subscribers_merged, left_on='IdSubscriber', right_on='IdOnPSX', how='inner')

In [59]:
len(final_df)

11124648

In [60]:
final_df.isna().sum()

IdSession       0
IdPSX           0
IdSubscriber    0
StartSession    0
EndSession      0
Duartion        0
UpTx            0
DownTx          0
IdClient        0
IdOnPSX         0
Status          0
Id              0
Contract        0
Documents       0
Email           0
IdPlan          0
Type            0
dtype: int64

In [61]:
final_df.to_csv('final_df.csv', index=False)

In [2]:
final_df = pd.read_csv('final_df.csv')
final_df

,IdSession,IdPSX,IdSubscriber,StartSession,EndSession,Duartion,UpTx,DownTx,IdClient,IdOnPSX,Status,Id,Contract,Documents,Email,IdPlan,Type
0,22324.0,5.0,6529.0,04-01-2024 01:35:35,04-01-2024 08:20:00,24265.0,1.184640e+10,1.185436e+10,30f5fd3e-f695-4193-b94c-3882be9eff83,6529,ON,30f5fd3e-f695-4193-b94c-3882be9eff83,GB81OITF05,internal.store.com/clients/documents/GB81OITF05,nmontoya@burnett-anderson.biz,3,P
1,20535.0,5.0,29242.0,03-01-2024 13:09:40,04-01-2024 08:17:12,68852.0,1.408189e+10,2.024532e+10,c66c5fd2-6e0b-467d-b1fd-404d521bc0c1,29242,ON,c66c5fd2-6e0b-467d-b1fd-404d521bc0c1,GB48BEPX77,internal.store.com/clients/documents/GB48BEPX77,matthew76@klein.biz,7,P
2,23238.0,5.0,29242.0,04-01-2024 08:18:52,04-01-2024 08:20:00,68.0,1.101986e+07,1.554654e+07,c66c5fd2-6e0b-467d-b1fd-404d521bc0c1,29242,ON,c66c5fd2-6e0b-467d-b1fd-404d521bc0c1,GB48BEPX77,internal.store.com/clients/documents/GB48BEPX77,matthew76@klein.biz,7,P
3,22857.0,5.0,79675.0,04-01-2024 05:27:17,04-01-2024 08:20:00,10363.0,9.411518e+08,1.038147e+09,feb18729-0e4e-401c-90d6-6488e71d9bc3,79675,ON,feb18729-0e4e-401c-90d6-6488e71d9bc3,GB43URPG41,internal.store.com/clients/documents/GB43URPG41,jasoncannon@cordova.org,5,P
4,22814.0,5.0,45133.0,04-01-2024 05:02:31,04-01-2024 08:20:00,11849.0,4.442024e+09,6.250973e+09,e2b0925b-c318-4b99-8e92-926c23cf56b7,45133,ON,e2b0925b-c318-4b99-8e92-926c23cf56b7,GB84YIIO74,internal.store.com/clients/documents/GB84YIIO74,mirandamichael@taylor-bell.org,5,P
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11124643,22775.0,2.0,15606.0,05-01-2024 02:21:26,04-01-2024 16:21:26,8914.0,2.574876e+09,3.719875e+09,a3d0a828-1776-4da2-911e-cf9d4637bf6d,15606,ON,a3d0a828-1776-4da2-911e-cf9d4637bf6d,GB11YBKD25,internal.store.com/clients/documents/GB11YBKD25,osbornekevin@ray.info,3,P
11124644,21244.0,2.0,39993.0,04-01-2024 17:35:14,04-01-2024 07:35:14,40486.0,2.980798e+10,2.988357e+10,55cd4aef-3d04-4965-abde-73d6e2a56481,39993,ON,55cd4aef-3d04-4965-abde-73d6e2a56481,GB44NJFN03,internal.store.com/clients/documents/GB44NJFN03,tonyalopez@young.com,1,P
11124645,21992.0,2.0,54026.0,04-01-2024 21:55:42,04-01-2024 11:55:42,24858.0,1.170152e+09,1.183539e+09,5e8e5855-68b9-4e17-b6bd-cf4c98acfbcf,54026,ON,5e8e5855-68b9-4e17-b6bd-cf4c98acfbcf,GB15OFVE54,internal.store.com/clients/documents/GB15OFVE54,stevensoncorey@barrera-mcdowell.org,5,P
11124646,19853.0,2.0,19194.0,04-01-2024 09:44:43,03-01-2024 23:44:43,68717.0,1.477835e+10,1.700411e+10,730ec055-748c-442d-9dc5-4e637eef5bd9,19194,ON,730ec055-748c-442d-9dc5-4e637eef5bd9,GB43JIEL50,internal.store.com/clients/documents/GB43JIEL50,davidford@hall.com,0,P


In [3]:
len(final_df.drop_duplicates())

11124648